In [ ]:
# !pip install langchain-community pypdf
# !pip install langchain-huggingface
# !pip install sentence-transformers
# !pip install -qU langchain-chroma

In [ ]:
import re
import hashlib

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


In [ ]:

def deterministic_chunk_id(source, page, chunk_text):
    payload = f"{source}|{page}|{chunk_text.strip().lower()}"
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


In [ ]:
"""Carrega o PDF e enriquece os metadados de cada página."""

file_path = "/home/senacgoon.local/202473567/Documents/senac_ia/uc15/senac_ia_uc15_nlp_project/data/pablo/sindilojas_2025_2026.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()
# docs = enrich_metadata(docs, file_path)

print(f"Páginas carregadas: {len(docs)}")
print("Exemplo de metadata:", docs[0].metadata)


In [ ]:
"""Split the loaded documents into smaller chunks and print the total number of chunks created."""

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600, chunk_overlap=80, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))

# all_splits = structural_then_char_split(docs, chunk_size=600, chunk_overlap=80)

# print(f"Total de chunks: {len(all_splits)}")
# print("Exemplo de metadata do chunk:", all_splits[0].metadata)

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
# embeddings = HuggingFaceEmbeddings(model_name="embaas/sentence-transformers-multilingual-e5-base")

In [ ]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

In [ ]:
vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

In [ ]:
chunk_ids = [
    deterministic_chunk_id(
        chunk.metadata["source"],
        chunk.metadata["page"],
        chunk.page_content
    )
    for chunk in all_splits
]

ids = vector_store.add_documents(documents=all_splits, ids=chunk_ids)

In [ ]:
results = vector_store.similarity_search_with_score(
    "Quais as garantias ao retornar de férias?"
)

doc, score = results[0]
print(f"Score: {score}\n")
print(doc)